

### 1. Error Detection Categories: 

---

**TOX (Toxicity):**  
Deviation in toxicity (hate, violence, or profanity) against an individual or group (e.g., religion, race, gender).  
*Why critical?* Toxic content can cause harm, incite violence or cause hate between different groups of people, which could lead to serious conflicts.

*Example 1:*  
- Source: "The president is a greatly admired leader."
- Back-translated MT could be: "The president is a greatly dreaded leader." 
- Reason: (Causing hate or affecting certain groups or more if translated incorrectly)

*Example 2:*  
- source: “Stop wasting valuable Internet resources (you’re hurting the planet)”
- Back-translated MT: “Stop wasting valuable Internet resources (may the  planet kill you)”
- Reason: critical mistranslation into violent content.

---

**SAF (Health or Safety Risks):**  
Deviation in health or safety risks, where translation errors may endanger the reader.  
*Why critical?* Incorrect health or safety information can lead to physical harm or dangerous actions.

*Example 1:*  
- Source: "Take two pills daily."  
- Back-translated MT: "Take twenty pills daily." (overdose affecting the patient life)

*Example 2:*  
- Source: “Wash your hands, or you will catch the coronavirus”
- Back-translated MT: “Shake hands, or you will catch the coronavirus”
- Reason: critical mistranslation, adding a safety risk.


---

**NAM (Named Entities):**  
Deviation in named entities (people, organizations, locations, etc.), such as deletion, mistranslation by another NE or common word. Or translated where the transliteration makes no sense in the target language.
*Why critical?* Misidentifying entities can cause confusion, legal issues, or misdirected actions.

*Example 1:*  
- Source: "Professor Smith, your supervisor, lives in Berlin. Go there and call him to pick you up"  
- Back-translated MT: "Professor Smith, your supervisor, lives in Stockholm. Go there and call him to pick you up" (Incorrect NE used, which could cause great confusion, waste time/money/effort.)

*Example 2:*  

- Source: “They are the worst band!”
- Back-translated MT: “Cold Play are the worst band!”
- Reason: critical introduction of named entity.
---

**SEN (Sentiment Polarity or Negation):**  
Deviation in sentiment or negation, where the translation reverses or alters the intended sentiment (removes/add/ negation or alter confidence rate).  
*Why critical?* Misrepresenting sentiment can damage reputations, relationships, or lives.  

*Example 1:*  
- Source: "The results are not promising."
- Back-translated MT: "The results are promising." (Negation removed, introducing false/misleading facts, e.g. percentage of a surgery success rate low becoming high or vice-versa) 

*Example 2:*  

- Source: “I never wrote this article, I just edited it”  
- Back-translated MT: “I never wrote this article, I never edited it”  
- Reason: critical mistranslation, the second clause was negated in the translation.
---

**NUM (Units/Time/Date/Numbers):**  
Deviation in numbers, units, dates, or times, such as incorrect translation or omission.  
*Why critical?* Errors can lead to missed appointments, financial loss, or logistical failures.  

*Example 1:*  
- Source: "The meeting is on July 10 at 3 PM."  
- Back-translated MT: "The meeting is on July 11 at 8 PM." (Incorrect date and time, which could cause great confusion, waste time/money/effort.)

*Example 2:*  
- Source: “From that point, turn right and drive 20 kilometers”
- Back-translated MT: “From that point, turn right and drive 20 miles”
- Reason: critical mistranslation as it leads to incorrect directions.

### Installing required Env:

In [1]:
pip install -U scikit-learn transformers pandas 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install --upgrade transformers[torch] accelerate


  Using cached accelerate-1.8.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.8.1-py3-none-any.whl (365 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch 

# Sanity Check for device usage
torch.cuda.is_available()
torch.cuda.get_device_name()

'NVIDIA GeForce GTX 1060 with Max-Q Design'

### Preprocessing and data prep 

In [ ]:
import pandas as pd

# Load and concatenate train/dev files for all 4 language pairs
lang_prefixes = ["encs", "ende", "enja", "enzh"]
train_dfs = []
dev_dfs = []

for prefix in lang_prefixes:
    train_file = f"{prefix}_majority_train.tsv"
    dev_file = f"{prefix}_majority_dev.tsv"
    train_df_lang = pd.read_csv(train_file, sep="\t", header=None, names=["id", "source", "target", "scores", "label"])
    dev_df_lang = pd.read_csv(dev_file, sep="\t", header=None, names=["id", "source", "target", "scores", "label"])
    print(f"Loaded {train_file} with {len(train_df_lang)} examples")
    print(f"Loaded {dev_file} with {len(dev_df_lang)} examples")
    train_dfs.append(train_df_lang)
    dev_dfs.append(dev_df_lang)

train_df = pd.concat(train_dfs, ignore_index=True)
dev_df = pd.concat(dev_dfs, ignore_index=True)

label_map = {"NOT": 0, "ERR": 1}
train_df["label"] = train_df["label"].map(label_map)
dev_df["label"] = dev_df["label"].map(label_map)

# Prepare input texts and labels
train_texts = (train_df["source"] + " [SEP] " + train_df["target"]).tolist()
train_labels = train_df["label"].tolist()
dev_texts = (dev_df["source"] + " [SEP] " + dev_df["target"]).tolist()
dev_labels = dev_df["label"].tolist()

print("************************************")
print(f"Number of training examples: {len(train_texts)}")
print(f"Number of development examples: {len(dev_texts)}")

Loaded encs_majority_train.tsv with 7476 examples
Loaded encs_majority_dev.tsv with 1000 examples
Loaded ende_majority_train.tsv with 7878 examples
Loaded ende_majority_dev.tsv with 1000 examples
Loaded enja_majority_train.tsv with 7658 examples
Loaded enja_majority_dev.tsv with 1000 examples
Loaded enzh_majority_train.tsv with 6859 examples
Loaded enzh_majority_dev.tsv with 1000 examples
************************************
Number of training examples: 29871
Number of development examples: 4000


### Transform data into tokens to be an input to the model


In [16]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize datasets
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
dev_encodings = tokenizer(dev_texts, truncation=True, padding=True, max_length=128)

class ErrorDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ErrorDataset(train_encodings, train_labels)
dev_dataset = ErrorDataset(dev_encodings, dev_labels)

### Bert Model training 

In [18]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define compute_metrics for evaluation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,0.453000
100,0.431100
150,0.468900
200,0.403900
250,0.439700
300,0.438400
350,0.450600
400,0.396900
450,0.465800
500,0.433700


TrainOutput(global_step=3734, training_loss=0.37056223696160584, metrics={'train_runtime': 1852.1929, 'train_samples_per_second': 32.255, 'train_steps_per_second': 2.016, 'total_flos': 1978466832626688.0, 'train_loss': 0.37056223696160584, 'epoch': 2.0})

### Evaluating against the dev_set

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

preds = trainer.predict(dev_dataset)
y_true = dev_labels
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["No Critical Error", "Critical Error"]))


Evaluation Results: {'eval_loss': 0.4137357175350189, 'eval_accuracy': 0.8375, 'eval_f1': 0.375, 'eval_runtime': 39.6795, 'eval_samples_per_second': 100.808, 'eval_steps_per_second': 3.15, 'epoch': 2.0}
                   precision    recall  f1-score   support

No Critical Error       0.87      0.95      0.91      3322
   Critical Error       0.54      0.29      0.38       678

         accuracy                           0.84      4000
        macro avg       0.70      0.62      0.64      4000
     weighted avg       0.81      0.84      0.82      4000

